In [6]:

# CNTNAP2 LFP PROCESSING — Initialization

%pip -q install -U ONE-api ibllib iblatlas scipy

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import defaultdict
from one.api import ONE
from iblatlas.regions import BrainRegions
from brainbox.io.one import SpikeSortingLoader
from scipy.signal import welch

import numpy as np
import pandas as pd
import hashlib
import gc

one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    username="intbrainlab",
    password="international",
    silent=True
)

br = BrainRegions()

GENOTYPE = "Cntnap2"
PROJECT = "angelaki_mouseASD"

ROOT = Path(
    "/content/drive/MyDrive/Reanalysis/"
    "Our_Analysis/lfp/spontaneous/Cntnap2/processing"
)

ROOT.mkdir(parents=True, exist_ok=True)

INSERTION_DIR = ROOT / "per_insertion"
INSERTION_DIR.mkdir(exist_ok=True)

N_WINDOWS = 30
WINDOW_S = 2.0
GUARD_S = 0.5
TUKEY_K = 3.0
MIN_ANIMALS = 5

BANDS = {
    "Delta": (1, 4),
    "Theta": (4, 8),
    "Alpha-like": (8, 12),
    "Beta": (12, 30),
    "Low gamma": (30, 55),
    "High gamma": (65, 100)
}

REGIONS = [
    "DG", "CA1", "CA3", "SUB",
    "VISp", "MOs", "CP", "MRN", "APN"
]

print("Genotype:", GENOTYPE)
print("Output:", ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Genotype: Cntnap2
Output: /content/drive/MyDrive/Reanalysis/Our_Analysis/lfp/spontaneous/Cntnap2/processing


In [ ]:

# ============================================================
# STEP 2 — CNTNAP2 INVENTORY (CORRECTED)
# ============================================================

GENOTYPE = "Cntnap2"
PROJECT = "angelaki_mouseASD"

# In this IBL dataset, Cntnap2 subjects use the CSP prefix.
subjects = one.alyx.rest("subjects", "list")

candidate_subjects = sorted({
    str(s["nickname"])
    for s in subjects
    if str(s.get("nickname", "")).upper().startswith("CSP")
})

print("CSP candidate subjects:", len(candidate_subjects))
print(candidate_subjects)

inventory_rows = []
project_subjects = set()

for i, subject in enumerate(candidate_subjects, 1):

    eids = one.search(
        project=PROJECT,
        subject=subject
    )

    if not len(eids):
        continue

    project_subjects.add(subject)

    print(
        f"[{i}/{len(candidate_subjects)}] "
        f"{subject}: {len(eids)} sessions"
    )

    for eid_i in eids:
        eid_i = str(eid_i)

        info = one.alyx.rest(
            "sessions", "read", id=eid_i
        )

        if str(info.get("subject")) != subject:
            continue

        insertions = one.alyx.rest(
            "insertions",
            "list",
            session=eid_i
        )

        paths = [
            str(p) for p in one.list_datasets(eid_i)
        ]

        for ins in insertions:
            probe = str(ins["name"])

            lf_files = [
                p for p in paths
                if p.startswith(
                    f"raw_ephys_data/{probe}/"
                )
                and p.endswith(".lf.cbin")
            ]

            inventory_rows.append({
                "subject": subject,
                "genotype": GENOTYPE,
                "eid": eid_i,
                "pid": str(ins["id"]),
                "probe": probe,
                "date": info.get("start_time"),
                "lf_available": bool(lf_files),
                "lf_files": len(lf_files)
            })

inventory_all = pd.DataFrame(inventory_rows)

if inventory_all.empty:
    raise RuntimeError(
        "No CSP insertions found in the project."
    )

inventory = (
    inventory_all.loc[
        inventory_all["lf_available"]
    ]
    .drop_duplicates("pid")
    .reset_index(drop=True)
)

# Preserve full and LF-only inventories.
inventory_all.to_csv(
    ROOT / "Cntnap2_LFP_inventory_all.csv",
    index=False
)

inventory.to_csv(
    ROOT / "Cntnap2_LFP_inventory.csv",
    index=False
)

print("\n" + "=" * 50)
print("CNTNAP2 INVENTORY")
print("=" * 50)

print("Project CSP subjects:", len(project_subjects))
print("Animals with LF:", inventory["subject"].nunique())
print("Sessions:", inventory["eid"].nunique())
print("LF insertions:", len(inventory))

display(inventory.head(10))

# Verify the LF cohort, not all CSP subjects in Alyx.
n_animals = inventory["subject"].nunique()
n_insertions = len(inventory)

print(f"Cntnap2 animals with LF: {n_animals}")
print(f"LF insertions: {n_insertions}")

if n_animals == 0 or n_insertions == 0:
    raise RuntimeError("No Cntnap2 LF data available.")

print("Inventory ready for processing.")


CSP candidate subjects: 25
['CSP001', 'CSP003', 'CSP004', 'CSP005', 'CSP006', 'CSP009', 'CSP010', 'CSP011', 'CSP012', 'CSP013', 'CSP015', 'CSP016', 'CSP017', 'CSP018', 'CSP022', 'CSP023', 'CSP025', 'CSP026', 'CSP027', 'CSP028', 'CSP029', 'CSP030', 'CSP031', 'CSP032', 'CSP033']
[1/25] CSP001: 75 sessions
[2/25] CSP003: 71 sessions
